# **MEDFIT AI**

Step-1: Install Required libraries

In [4]:
# ===========================
# Install Required Libraries
# ===========================

!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-groq \
langchain-huggingface \
chromadb \
pymupdf \
pypdf \
sentence-transformers \
langchain-text-splitters \
langchain-chroma

Verification

In [5]:
import langchain

print("LangChain Version:", langchain.__version__)

LangChain Version: 1.3.13


Step 2: Import required libraries

In [6]:
# ===========================
# Import Required Libraries
# ===========================

import os
import json
import warnings

warnings.filterwarnings("ignore")

# LangChain Document
from langchain_core.documents import Document

# PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chroma Vector Database
from langchain_chroma import Chroma

# Groq LLM
from langchain_groq import ChatGroq

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# Output Parser
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough, RunnableLambda



print("All libraries imported successfully!")

All libraries imported successfully!


In [7]:
# ===========================
# Configure API Keys
# ===========================

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("GROQ API KEY: ")
print("API Keys Loaded Successfully!")

GROQ API KEY: ··········
API Keys Loaded Successfully!


Verify

In [8]:
# ===========================
# Initialize Groq LLM
# ===========================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response = llm.invoke("Hello!")

print(response.content)

Hello. How can I help you today?


Step-4: Load JSON files

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os

json_folder = "/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset"

In [11]:
json_files = [
    os.path.join(json_folder, file)
    for file in os.listdir(json_folder)
    if file.endswith(".json")
]

print("JSON Files Found:")
for file in json_files:
    print(file)

JSON Files Found:
/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset/Medical_Dataset.json
/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset/Sports_Dataset.json
/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset/Physiotherapy_Dataset.json
/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset/Yoga_Dataset.json
/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/JSON dataset/Nutrition_Dataset.json


In [12]:
import json

json_data = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)

        json_data.extend(data)

print("Total JSON Records:", len(json_data))

Total JSON Records: 625


Verify

In [13]:
json_data[0]

{'id': 1,
 'dataset': 'Medical_Dataset',
 'category': 'Cardiology',
 'subtopic': 'Heart Anatomy',
 'question': 'What are the main functions of the human heart?',
 'answer': 'The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health.',
 'keywords': ['heart',
  'cardiology',
  'blood circulation',
  'heart anatomy',
  'cardiovascular system'],
 'difficulty': 'Beginner',
 'related_topics': ['Blood Circulation',
  'Heart Chambers',
  'Cardiovascular System'],
 'precautions': 'Individuals experiencing chest pain, dizziness, or shortness of breath should seek immediate medical evalu

Step 5: Load pdf files

In [14]:
pdf_folder = "/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF dataset"

In [15]:
pdf_files = [
    os.path.join(pdf_folder, file)
    for file in os.listdir(pdf_folder)
    if file.endswith(".pdf")
]

print(pdf_files)

['/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF dataset/Medical Health Pdf.pdf']


In [16]:
# ==========================================
# Step 5 : Load PDF Files (Correct Version)
# ==========================================

import os
from langchain_community.document_loaders import PyMuPDFLoader

pdf_documents = []

print("=" * 80)
print("Loading PDF Files")
print("=" * 80)

for pdf_path in pdf_files:

    print(f"\nLoading: {os.path.basename(pdf_path)}")

    loader = PyMuPDFLoader(pdf_path)

    docs = loader.load()

    print(f"Total Pages: {len(docs)}")

    pdf_documents.extend(docs)

print("\n" + "=" * 80)
print("Total Pages Loaded:", len(pdf_documents))
print("=" * 80)

Loading PDF Files

Loading: Medical Health Pdf.pdf
Total Pages: 16

Total Pages Loaded: 16


In [17]:
print("=" * 80)

for i, doc in enumerate(pdf_documents):

    print(f"Document {i+1}")

    print("Source :", os.path.basename(doc.metadata["source"]))

    print("Page :", doc.metadata["page"] + 1)

    print("Characters :", len(doc.page_content))

    print("-" * 80)

Document 1
Source : Medical Health Pdf.pdf
Page : 1
Characters : 1605
--------------------------------------------------------------------------------
Document 2
Source : Medical Health Pdf.pdf
Page : 2
Characters : 1526
--------------------------------------------------------------------------------
Document 3
Source : Medical Health Pdf.pdf
Page : 3
Characters : 1595
--------------------------------------------------------------------------------
Document 4
Source : Medical Health Pdf.pdf
Page : 4
Characters : 1331
--------------------------------------------------------------------------------
Document 5
Source : Medical Health Pdf.pdf
Page : 5
Characters : 1686
--------------------------------------------------------------------------------
Document 6
Source : Medical Health Pdf.pdf
Page : 6
Characters : 1712
--------------------------------------------------------------------------------
Document 7
Source : Medical Health Pdf.pdf
Page : 7
Characters : 2296
------------------------

In [ ]:
from collections import Counter

page_counter = Counter()

for doc in pdf_documents:
    page_counter[os.path.basename(doc.metadata["source"])] += 1

print(page_counter)

Counter({'Medical Health Pdf.pdf': 16})


In [18]:
text_documents = []
empty_documents = []

for doc in pdf_documents:

    if doc.page_content.strip():

        text_documents.append(doc)

    else:

        empty_documents.append(doc)

print("Text Pages :", len(text_documents))
print("Empty Pages :", len(empty_documents))

Text Pages : 16
Empty Pages : 0


Step 6: Convert JSON Records into LangChain Documents

In [19]:
from langchain_core.documents import Document

json_documents = []

for record in json_data:

    text = ""

    for key, value in record.items():

        if isinstance(value, list):

            value = ", ".join(map(str, value))

        text += f"{key}: {value}\n"

    document = Document(

        page_content=text,

        metadata={

            "source": "JSON",

            "category": record.get("category", "General"),

            "dataset": record.get("dataset","Unknown"),

            "type": "JSON"

        }

    )

    json_documents.append(document)

print("JSON Documents:", len(json_documents))

JSON Documents: 625


In [20]:
json_documents[0]

Document(metadata={'source': 'JSON', 'category': 'Cardiology', 'dataset': 'Medical_Dataset', 'type': 'JSON'}, page_content='id: 1\ndataset: Medical_Dataset\ncategory: Cardiology\nsubtopic: Heart Anatomy\nquestion: What are the main functions of the human heart?\nanswer: The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health.\nkeywords: heart, cardiology, blood circulation, heart anatomy, cardiovascular system\ndifficulty: Beginner\nrelated_topics: Blood Circulation, Heart Chambers, Cardiovascular System\nprecautions: Individuals experiencing chest pain, dizziness, or shortn

In [21]:
documents = json_documents + pdf_documents

print("Total Documents:", len(documents))

Total Documents: 641


verfiy

In [22]:
print(documents[0])

print("="*100)

print(documents[-1])

page_content='id: 1
dataset: Medical_Dataset
category: Cardiology
subtopic: Heart Anatomy
question: What are the main functions of the human heart?
answer: The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health.
keywords: heart, cardiology, blood circulation, heart anatomy, cardiovascular system
difficulty: Beginner
related_topics: Blood Circulation, Heart Chambers, Cardiovascular System
precautions: Individuals experiencing chest pain, dizziness, or shortness of breath should seek immediate medical evaluation.
consultation: Consult a qualified doctor or healthcare professi

Step 7: Clean the documents

In [23]:
# ==========================================
# Step 7 : Clean Documents
# ==========================================

import re

def clean_text(text):
    """
    Clean extracted text by removing extra spaces,
    tabs, and multiple newlines.
    """

    text = re.sub(r"\n+", "\n", text)

    text = re.sub(r"\t+", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [24]:
cleaned_documents = []

for doc in documents:

    cleaned_text = clean_text(doc.page_content)

    cleaned_documents.append(
        Document(
            page_content=cleaned_text,
            metadata=doc.metadata
        )
    )

print("Total Cleaned Documents :", len(cleaned_documents))

Total Cleaned Documents : 641


In [25]:
print(cleaned_documents[0].page_content[:700])

id: 1 dataset: Medical_Dataset category: Cardiology subtopic: Heart Anatomy question: What are the main functions of the human heart? answer: The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health. keywords: heart, cardiology, blood circulation, heart anatomy, cardiovascular syst


Step-8: Enhance metadata

In [26]:
# ==========================================
# Step 8 : Metadata Enhancement
# ==========================================

enhanced_documents = []

for i, doc in enumerate(cleaned_documents):

    metadata = doc.metadata.copy()

    metadata["document_id"] = i + 1

    metadata["length"] = len(doc.page_content)

    metadata["source"] = metadata.get("source", "Unknown")

    metadata["document_type"] = metadata.get("type", "PDF")

    enhanced_documents.append(

        Document(

            page_content=doc.page_content,

            metadata=metadata

        )

    )
print(enhanced_documents[0].metadata)
print("Metadata Added Successfully")

{'source': 'JSON', 'category': 'Cardiology', 'dataset': 'Medical_Dataset', 'type': 'JSON', 'document_id': 1, 'length': 1067, 'document_type': 'JSON'}
Metadata Added Successfully


In [27]:
enhanced_documents[0].metadata

{'source': 'JSON',
 'category': 'Cardiology',
 'dataset': 'Medical_Dataset',
 'type': 'JSON',
 'document_id': 1,
 'length': 1067,
 'document_type': 'JSON'}

Step 9: Chunk the documents

In [28]:
# ==========================================
# Step 9 : Chunk Documents
# ==========================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        " ",
        ""
    ]
)

In [29]:
chunked_documents = text_splitter.split_documents(
    enhanced_documents
)

print("Total Chunks :", len(chunked_documents))

Total Chunks : 1056


In [30]:
chunked_documents[0]

Document(metadata={'source': 'JSON', 'category': 'Cardiology', 'dataset': 'Medical_Dataset', 'type': 'JSON', 'document_id': 1, 'length': 1067, 'document_type': 'JSON'}, page_content='id: 1 dataset: Medical_Dataset category: Cardiology subtopic: Heart Anatomy question: What are the main functions of the human heart? answer: The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health')

In [31]:
chunked_documents[0].metadata

{'source': 'JSON',
 'category': 'Cardiology',
 'dataset': 'Medical_Dataset',
 'type': 'JSON',
 'document_id': 1,
 'length': 1067,
 'document_type': 'JSON'}

Step 10: Generate Embeddings & Create ChromaDB

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(

    model_name="BAAI/bge-small-en-v1.5",

    model_kwargs={
        "device":"cpu"
    },

    encode_kwargs={
        "normalize_embeddings":True
    }

)


print("BGE Embedding Model Loaded Successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE Embedding Model Loaded Successfully


Create chroma db

In [33]:
!pip install langchain-chroma chromadb

In [34]:
import shutil
import os
chroma_path="./database/chroma"

if not os.path.exists(chroma_path):

    vector_db = Chroma.from_documents(
        documents=chunked_documents,
        embedding=embedding_model,
        persist_directory=chroma_path
    )

else:

    vector_db = Chroma(
        persist_directory=chroma_path,
        embedding_function=embedding_model
    )

print("ChromaDB Created successfully")

ChromaDB Created successfully


Step 11: Create the Retriever

In [35]:
# ==========================================
# Step 11 : Create Retriever
# ==========================================

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever Created Successfully!")

Retriever Created Successfully!


11.2 (Test)

In [36]:
query = "What are the symptoms of diabetes?"

docs = retriever.invoke(query)

print("Retrieved Chunks :", len(docs))

Retrieved Chunks : 5


11.3 (Test)

In [37]:
for i, doc in enumerate(docs):

    print("="*80)

    print(f"Chunk {i+1}")

    print(doc.page_content[:500])

    print("\nMetadata:")

    print(doc.metadata)

Chunk 1
. Symptoms Symptoms of diabetes may occur suddenly. In type 2 diabetes, the symptoms can be mild and may take many years to be noticed. Symptoms of diabetes include: • feeling very thirsty • needing to urinate more often than usual • blurred vision • feeling tired • losing weight unintentionally Over time, diabetes can damage blood vessels in the heart, eyes, kidneys and nerves. People with diabetes have a higher risk of health problems including heart attack, stroke and kidney failure. Diabetes

Metadata:
{'total_pages': 16, 'creator': 'Microsoft® Word 2024', 'subject': '', 'creationDate': "D:20260723213554+05'30'", 'trapped': '', 'moddate': '2026-07-23T21:35:54+05:30', 'title': '', 'author': 'PARTH KHERA', 'document_type': 'PDF', 'file_path': '/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF dataset/Medical Health Pdf.pdf', 'creationdate': '2026-07-23T21:35:54+05:30', 'keywords': '', 'modDate': "D:20260723213554+05'30'", 'producer': 'Microsoft® Word 2024', 'sourc

Step 12: create prompt template

In [60]:
# ==========================================
# Step 12 : Prompt Template
# ==========================================

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are MedFit AI, an AI-powered healthcare assistant specializing in:
- Medical Conditions
- Sports Injuries
- Physiotherapy
- Yoga
- Nutrition

You must strictly follow these rules:

1. Answer ONLY using the provided context.
2. Never use your own knowledge.
3. Never guess, infer, or hallucinate information.
4. If the required information is NOT present in the provided context, reply EXACTLY with:

"I couldn't find this information in my knowledge base."

5. If the information is not found, DO NOT add any explanation, disclaimer, apology, or doctor consultation note. Return ONLY the this sentence:
"I couldn't find this information in my knowledge base."


6. If the information IS found in the provided context, provide a clear, accurate, and well-structured answer with the following:
NOTE:
For further information, proper diagnosis, and appropriate treatment, please consult a qualified healthcare professional.

Context:
{context}

Question:
{question}

Answer:
""")

print("✅ Prompt Created Successfully!")

✅ Prompt Created Successfully!


In [53]:
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are MedFit AI, an AI-powered healthcare assistant specializing in:\n- Medical Conditions\n- Sports Injuries\n- Physiotherapy\n- Yoga\n- Nutrition\n\nYou must strictly follow these rules:\n\n1. Answer ONLY using the provided context.\n2. Do NOT use your own knowledge.\n3. Do NOT guess, infer, or hallucinate information.\n4. If the answer is not available in the provided context, reply EXACTLY with:\n"I couldn\'t find this information in my knowledge base."\n5. If the answer is not found, DO NOT add any additional text, explanation, disclaimer, or doctor consultation note.\n6. If the answer is found, provide a clear, accurate, and well-structured response.\n7. After every successful answer, whose answer is found from the database add the following on a new line 

Step 13: Build RAG chain

In [61]:
# vector_db = Chroma(

#     persist_directory="./database/chroma",

#     embedding_function=embedding_model

# )


# retriever = vector_db.as_retriever(
#     search_kwargs={
#         "k":5
#     }
# )


def format_docs(docs):

    return "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in docs
    )


rag_chain = (

    {
        "context": retriever | RunnableLambda(format_docs),

        "question": RunnablePassthrough()
    }

    | prompt
    | llm
    | StrOutputParser()

)

In [63]:
question = "What are the symptoms of hypertension?"

response = rag_chain.invoke(question)

print(response)

The symptoms of hypertension can vary, but people with very high blood pressure (usually 180/120 or higher) can experience symptoms including: 
• severe headaches 
• chest pain 
• dizziness 
• difficulty breathing 
• nausea 
• vomiting 
• blurred vision or other vision changes 
• anxiety 
• confusion 
• buzzing in the ears 
• nosebleeds. 
However, most people with hypertension don’t feel any symptoms. 

NOTE: For further information, proper diagnosis, and appropriate treatment, please consult a qualified healthcare professional.


In [62]:
question = "What should I know about symptoms of Beginner Yoga?"

response = rag_chain.invoke(question)

print(response)

I couldn't find this information in my knowledge base.


In [ ]:
question = "What foods are rich in protein?"

response = rag_chain.invoke(question)

print(response)

High protein foods include eggs, chicken, fish, milk, yogurt, lentils, beans, tofu, nuts, and seeds. Including a variety of these foods helps meet daily protein requirements.

Please consult a doctor for further information and treatment.


In [64]:
question = "Who won the FIFA World Cup in 2022?"

response = rag_chain.invoke(question)

print(response)

I couldn't find this information in my knowledge base.


In [65]:
question = "Who won the Cricket World Cup in 2023?"

response = rag_chain.invoke(question)

print(response)

I couldn't find this information in my knowledge base.


In [66]:
question = "What is cobra pose?"

response = rag_chain.invoke(question)

print(response)

Cobra Pose, also known as Bhujangasana, is a yoga pose that strengthens the back muscles and gently stretches the chest and shoulders. It helps improve spinal flexibility, posture, and body awareness when performed correctly.

NOTE: For further information, proper diagnosis, and appropriate treatment, please consult a qualified healthcare professional.


In [67]:
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

Step 14: create chat function

In [68]:
# ==========================================
# Step 14 : Chat Function
# ==========================================

def ask_medfit_ai(question):
    """
    Ask a question to the RAG chatbot.
    Returns the generated answer.
    """

    try:

        response = rag_chain.invoke(question)

        return response

    except Exception as e:

        return f"Error : {str(e)}"

In [ ]:
question = "What are the symptoms of diabetes?"

answer = ask_medfit_ai(question)

print(answer)

The symptoms of diabetes include: feeling very thirsty, needing to urinate more often than usual, blurred vision, feeling tired, and losing weight unintentionally. Additionally, other symptoms mentioned in the context are numbness, burning pain, tingling, muscle weakness, and reduced sensation in the hands or feet, as well as excessive thirst, frequent urination, increased hunger, fatigue, blurred vision, and slow wound healing.

Please consult a doctor for further information and treatment.


In [ ]:
questions = [

    "What is hypertension?",

    "Suggest yoga for stress.",

    "Best foods rich in protein.",

    "How can physiotherapy help in knee pain?"

]

for q in questions:

    print("="*80)

    print("Question :", q)

    print()

    print(ask_medfit_ai(q))

    print()

Question : What is hypertension?

Hypertension (high blood pressure) is when the pressure in your blood vessels is too high (140/90 mmHg or higher). 

Please consult a doctor for further information and treatment.

Question : Suggest yoga for stress.

Gentle evening yoga, restorative yoga, yoga nidra, easy pose meditation, and gentle neck yoga are some of the yoga practices that can help reduce stress by releasing physical tension, calming the mind, and promoting relaxation.

Please consult a doctor for further information and treatment.

Question : Best foods rich in protein.

High protein foods include eggs, chicken, fish, milk, yogurt, lentils, beans, tofu, nuts, and seeds. Including a variety of these foods helps meet daily protein requirements.

Please consult a doctor for further information and treatment.

Question : How can physiotherapy help in knee pain?

I couldn't find this information in my knowledge base.

Please consult a doctor for further information and treatment.



Step 15: Display Sources with the Answer

In [69]:
# ==========================================
# Step 15 : Source Citation
# ==========================================

def ask_with_sources(question):

    docs = retriever.invoke(question)

    answer = rag_chain.invoke(question)

    sources = []

    for doc in docs:

        source = doc.metadata.get("source", "Unknown")

        if source not in sources:

            sources.append(source)

    return {

        "question": question,

        "answer": answer,

        "sources": sources

    }

In [ ]:
response = ask_with_sources(

    "What foods are rich in protein?"

)

print("Question")

print(response["question"])

print()

print("Answer")

print(response["answer"])

print()

print("Sources")

for s in response["sources"]:

    print("-", s)

Question
What foods are rich in protein?

Answer
High protein foods include eggs, chicken, fish, milk, yogurt, lentils, beans, tofu, nuts, and seeds. Including a variety of these foods helps meet daily protein requirements.

Please consult a doctor for further information and treatment.

Sources
- JSON


In [70]:
response = ask_with_sources(

    "What is depression?"

)

print(response["answer"])

print()

for doc in retriever.invoke("What is depression?"):

    print("="*70)

    print(doc.metadata)

    print()

    print(doc.page_content[:300])

I couldn't find this information in my knowledge base.

{'dataset': 'Medical_Dataset', 'category': 'Neurology', 'type': 'JSON', 'document_id': 132, 'document_type': 'JSON', 'length': 881, 'source': 'JSON'}

id: 132 dataset: Medical_Dataset category: Neurology subtopic: Migraine question: What is a migraine and how can it be managed? answer: Migraine is a neurological condition characterized by recurrent moderate to severe headaches, often accompanied by nausea, vomiting, and sensitivity to light or sou
{'type': 'JSON', 'document_id': 15, 'document_type': 'JSON', 'source': 'JSON', 'dataset': 'Medical_Dataset', 'length': 1045, 'category': 'Cardiology'}

id: 15 dataset: Medical_Dataset category: Cardiology subtopic: Heart Failure question: What is heart failure and how does it affect the body? answer: Heart failure occurs when the heart cannot pump blood efficiently enough to meet the body's needs. Symptoms often include fatigue, shortness of breath
{'category': 'Infectious Diseases', 'so

## TESTING


16.1 Basic Retrieval Testing

In [71]:
# ==========================================
# Step 16.1 : Retrieval Testing
# ==========================================

test_queries = [

    "What are symptoms of diabetes?",

    "How can heart disease be prevented?",

    "What are asthma symptoms?",

    "What causes hypertension?",

    "Benefits of yoga?"

]


for query in test_queries:

    print("="*100)

    print("QUERY:")
    print(query)

    docs = retriever.invoke(query)

    print("\nRetrieved Documents:", len(docs))

    for i,doc in enumerate(docs):

        print("\nChunk:", i+1)

        print(
            doc.page_content[:300]
        )

        print(
            "Metadata:",
            doc.metadata
        )

QUERY:
What are symptoms of diabetes?

Retrieved Documents: 5

Chunk: 1
. Symptoms Symptoms of diabetes may occur suddenly. In type 2 diabetes, the symptoms can be mild and may take many years to be noticed. Symptoms of diabetes include: • feeling very thirsty • needing to urinate more often than usual • blurred vision • feeling tired • losing weight unintentionally Ove
Metadata: {'creationDate': "D:20260723213554+05'30'", 'creator': 'Microsoft® Word 2024', 'modDate': "D:20260723213554+05'30'", 'subject': '', 'format': 'PDF 1.7', 'total_pages': 16, 'document_type': 'PDF', 'document_id': 631, 'moddate': '2026-07-23T21:35:54+05:30', 'page': 5, 'title': '', 'producer': 'Microsoft® Word 2024', 'length': 1673, 'trapped': '', 'creationdate': '2026-07-23T21:35:54+05:30', 'keywords': '', 'source': '/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF dataset/Medical Health Pdf.pdf', 'author': 'PARTH KHERA', 'file_path': '/content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF da

16.2 RAG Answer Testing

In [72]:
# ==========================================
# Step 16.2 : RAG Response Testing
# ==========================================


rag_test_questions = [

    "What are the symptoms of diabetes?",

    "How to control high blood pressure?",

    "What foods are rich in protein?",

    "How does physiotherapy help knee pain?",

    "What is cobra pose in yoga?",

    "Who won FIFA world cup 2022?"

]


for question in rag_test_questions:


    print("="*100)

    print("QUESTION:")

    print(question)


    answer = ask_medfit_ai(question)


    print("\nANSWER:")

    print(answer)


QUESTION:
What are the symptoms of diabetes?

ANSWER:
The symptoms of diabetes include: 
• feeling very thirsty 
• needing to urinate more often than usual 
• blurred vision 
• feeling tired 
• losing weight unintentionally 
• excessive thirst 
• frequent urination 
• increased hunger 
• fatigue 
• blurred vision 
• slow wound healing 
• numbness 
• burning pain 
• tingling 
• muscle weakness 
• reduced sensation in the hands or feet.

NOTE: For further information, proper diagnosis, and appropriate treatment, please consult a qualified healthcare professional.
QUESTION:
How to control high blood pressure?

ANSWER:
To control high blood pressure, you can make lifestyle changes such as: 
• eating a healthy, low-salt diet 
• losing weight 
• being physically active 
• quitting tobacco 
• reducing and managing stress 
• regularly checking blood pressure 
• treating high blood pressure 
• managing other medical conditions 
• eating more vegetables and fruits 
• sitting less 
• being more p

16.3 JSON Dataset Based Testing

In [ ]:
# ==========================================
# Step 16.3 : JSON Knowledge Base Testing
# ==========================================

import random


sample_records = random.sample(
    json_data,
    20
)


for record in sample_records:


    question = record.get(
        "question"
    )


    expected_answer = record.get(
        "answer"
    )


    print("="*100)

    print("QUESTION:")

    print(question)


    response = ask_medfit_ai(
        question
    )


    print("\nMODEL ANSWER:")

    print(response)


    print("\nEXPECTED ANSWER:")

    print(expected_answer)


QUESTION:
What is atrial fibrillation and why is it considered a serious condition?

MODEL ANSWER:
Atrial fibrillation (AF) is the most common sustained heart rhythm disorder, characterized by rapid and irregular electrical activity in the upper chambers of the heart. It can cause palpitations, fatigue, dizziness, and shortness of breath. AF increases the risk of stroke because blood may pool inside the atria and form clots. 

Please consult a doctor for further information and treatment.

EXPECTED ANSWER:
Atrial fibrillation (AF) is the most common sustained heart rhythm disorder, characterized by rapid and irregular electrical activity in the upper chambers of the heart. It can cause palpitations, fatigue, dizziness, and shortness of breath. AF increases the risk of stroke because blood may pool inside the atria and form clots. Treatment may include medications, electrical cardioversion, catheter ablation, or anticoagulant therapy depending on individual risk.
QUESTION:
What causes j

16.4 Category-wise Testing

In [ ]:
# ==========================================
# Step 16.4 : Category Testing
# ==========================================


from collections import defaultdict


category_questions = defaultdict(list)


for record in json_data:

    category = record.get(
        "category",
        "Unknown"
    )

    question = record.get(
        "question"
    )

    if question:

        category_questions[category].append(
            question
        )


for category, questions in category_questions.items():


    print("\n")
    print("="*100)

    print(
        "CATEGORY:",
        category
    )


    test_question = questions[0]


    print(
        "Question:",
        test_question
    )


    answer = ask_medfit_ai(
        test_question
    )


    print(
        "\nAnswer:"
    )

    print(answer)




CATEGORY: Cardiology
Question: What are the main functions of the human heart?

Answer:
The human heart is a muscular organ responsible for pumping oxygen-rich blood throughout the body and returning oxygen-poor blood to the lungs. It has four chambers that work together to maintain continuous blood circulation. Proper heart function ensures that organs receive adequate oxygen and nutrients while waste products are removed efficiently. Maintaining a healthy lifestyle with regular exercise, a balanced diet, and avoiding tobacco supports optimal heart health.

Please consult a doctor for further information and treatment.


CATEGORY: Endocrinology
Question: What is diabetes mellitus and what are its common symptoms?

Answer:
Diabetes mellitus is a condition in which blood sugar remains abnormally high because the body does not produce enough insulin or cannot use it effectively. Common symptoms include excessive thirst, frequent urination, increased hunger, fatigue, blurred vision, and

16.5 Unknown Question / Hallucination Testing

In [ ]:
# ==========================================
# Step 16.5 : Hallucination Test
# ==========================================


unknown_questions = [

    "Who won FIFA World Cup 2026?",

    "What is quantum physics?",

    "Who is the Prime Minister of Japan?",

    "Explain blockchain technology"

]


for question in unknown_questions:


    print("="*100)

    print("QUESTION:")

    print(question)


    answer = ask_medfit_ai(
        question
    )


    print("\nANSWER:")

    print(answer)


QUESTION:
Who won FIFA World Cup 2026?

ANSWER:
I couldn't find this information in my knowledge base. 
Please consult a doctor for further information and treatment.
QUESTION:
What is quantum physics?

ANSWER:
I couldn't find this information in my knowledge base. 
Please consult a doctor for further information and treatment.
QUESTION:
Who is the Prime Minister of Japan?

ANSWER:
I couldn't find this information in my knowledge base. 
Please consult a doctor for further information and treatment.
QUESTION:
Explain blockchain technology

ANSWER:
I couldn't find this information in my knowledge base. 
Please consult a doctor for further information and treatment.


16.6 Source Verification Testing

Check whether retrieved sources are meaningful.

In [ ]:
# ==========================================
# Step 16.6 : Source Testing
# ==========================================


query="What are diabetes symptoms?"


result = ask_with_sources(
    query
)


print("Question:")
print(result["question"])


print("\nAnswer:")
print(result["answer"])


print("\nSources:")

for source in result["sources"]:

    print(
        "-",
        source
    )


Question:
What are diabetes symptoms?

Answer:
Symptoms of diabetes include: feeling very thirsty, needing to urinate more often than usual, blurred vision, feeling tired, and losing weight unintentionally. Additionally, common symptoms include excessive thirst, frequent urination, increased hunger, fatigue, blurred vision, and slow wound healing. 

Please consult a doctor for further information and treatment.

Sources:
- /content/drive/MyDrive/DataSets Colab File/MedFit Dataset/PDF dataset/Medical Health Pdf.pdf
- JSON


16.7 Retrieval Score Testing

In [ ]:
# ==========================================
# Step 16.7 : Simple Retrieval Accuracy
# ==========================================


def keyword_test(
    question,
    keywords
):


    docs = retriever.invoke(
        question
    )


    retrieved_text = " ".join(
        [
            doc.page_content.lower()
            for doc in docs
        ]
    )


    matched=[]


    for word in keywords:

        if word.lower() in retrieved_text:

            matched.append(word)


    score = len(matched)/len(keywords)


    return score, matched



score, matched = keyword_test(

    "What are symptoms of diabetes?",

    [
        "diabetes",
        "blood",
        "sugar",
        "thirst"
    ]

)


print(
    "Retrieval Score:",
    score
)


print(
    "Matched:",
    matched
)


Retrieval Score: 1.0
Matched: ['diabetes', 'blood', 'sugar', 'thirst']


16.8 Complete Test Report Generator

In [73]:
# ==========================================
# Step 16.8 : Test Summary
# ==========================================


test_cases = [

("Diabetes symptoms",
"What are symptoms of diabetes?"),

("Heart disease",
"How to prevent heart disease?"),

("Protein foods",
"What foods are rich in protein?"),

("Yoga",
"What is cobra pose?"),

("Knee pain",
"How physiotherapy helps knee pain?")

]


results=[]


for name,question in test_cases:


    answer = ask_medfit_ai(
        question
    )


    results.append(
        {
            "Test Case":name,
            "Question":question,
            "Answer":answer[:200]
        }
    )


import pandas as pd


test_report=pd.DataFrame(results)


test_report


,Test Case,Question,Answer
0,Diabetes symptoms,What are symptoms of diabetes?,Symptoms of diabetes include: \n• feeling very...
1,Heart disease,How to prevent heart disease?,Preventing heart disease involves maintaining ...
2,Protein foods,What foods are rich in protein?,"High protein foods include eggs, chicken, fish..."
3,Yoga,What is cobra pose?,"Cobra Pose, also known as Bhujangasana, is a y..."
4,Knee pain,How physiotherapy helps knee pain?,I couldn't find this information in my knowled...


# **CHATBOT DEMO**

17.1 Single User Query Testing

In [ ]:
# ==========================================
# Step 17.1 : Interactive MedFit AI Chat
# ==========================================


user_query = input(
    "Ask MedFit AI your question: "
)


answer = ask_medfit_ai(
    user_query
)


print("\n" + "="*80)

print("USER QUERY:")
print(user_query)


print("\nMEDFIT AI ANSWER:")

print(answer)

print("="*80)

Ask MedFit AI your question: What are the symptoms of hypertension?

USER QUERY:
What are the symptoms of hypertension?

MEDFIT AI ANSWER:
Most people with hypertension don’t feel any symptoms. However, very high blood pressures can cause symptoms including: severe headaches, chest pain, dizziness, difficulty breathing, nausea, vomiting, blurred vision or other vision changes, anxiety, confusion, buzzing in the ears, and nosebleeds.

Please consult a doctor for further information and treatment.


17.2 Multiple Conversation Testing

In [74]:
# ==========================================
# Step 17.2 : Continuous Chat Mode
# ==========================================


print("MedFit AI Chatbot")
print("Type 'exit' to stop\n")


while True:

    user_query = input(
        "You: "
    )


    if user_query.lower()=="exit":

        print(
            "MedFit AI: Thank you for using MedFit AI."
        )

        break


    answer = ask_medfit_ai(
        user_query
    )


    print("\nMedFit AI:")

    print(answer)

    print("\n"+"-"*80)

MedFit AI Chatbot
Type 'exit' to stop

You: what is yoga?

MedFit AI:
I couldn't find this information in my knowledge base.

--------------------------------------------------------------------------------
You: what is cobra pose?

MedFit AI:
Cobra Pose, also known as Bhujangasana, is a yoga pose that strengthens the back muscles and gently stretches the chest and shoulders. It helps improve spinal flexibility, posture, and body awareness when performed correctly.

NOTE: For further information, proper diagnosis, and appropriate treatment, please consult a qualified healthcare professional.

--------------------------------------------------------------------------------
You: what are the symptoms of heart disease?

MedFit AI:
Common symptoms of heart disease include chest pain or discomfort, shortness of breath, fatigue, palpitations, dizziness, swelling of the legs, and pain that may radiate to the jaw, shoulder, or left arm. Symptoms vary depending on the underlying condition and m

In [ ]:
query = "What are symptoms of diabetes?"

answer = ask_medfit_ai(query)

print(answer)

The symptoms of diabetes include: feeling very thirsty, needing to urinate more often than usual, blurred vision, feeling tired, and losing weight unintentionally. Additionally, symptoms can also include numbness, burning pain, tingling, muscle weakness, and reduced sensation in the hands or feet, especially in the case of diabetic neuropathy.

Please consult a doctor for further information and treatment.


In [75]:
# ==========================================
# Step 17.1 : Interactive MedFit AI Chat
# ==========================================


user_query = input(
    "Ask MedFit AI your question: "
)


answer = ask_medfit_ai(
    user_query
)


print("\n" + "="*80)

print("USER QUERY:")

print(user_query)


print("\nMEDFIT AI ANSWER:")

print(answer)

print("="*80)

Ask MedFit AI your question: who won fifa?

USER QUERY:
who won fifa?

MEDFIT AI ANSWER:
I couldn't find this information in my knowledge base.


In [76]:
!pip install -q gradio

In [77]:
import gradio as gr


def medfit_chat(message, history):
    """
    Chat function for Gradio
    """

    try:
        answer = ask_medfit_ai(message)

    except Exception as e:
        answer = f"Error: {str(e)}"

    history.append((message, answer))

    return history, history


with gr.Blocks(title="MedFit AI") as demo:

    gr.Markdown(
        """
        # 🏥 MedFit AI

        Medical RAG Chatbot

        Ask questions related to:

        • Medical Diseases

        • Nutrition

        • Physiotherapy

        • Sports Injuries

        • Yoga

        """
    )

    chatbot = gr.Chatbot(
        height=500,
        label="Conversation"
    )

    msg = gr.Textbox(
        placeholder="Ask your medical question...",
        label="Question"
    )

    clear = gr.Button("Clear Chat")

    msg.submit(
        medfit_chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, chatbot]
    )

    clear.click(
        lambda: [],
        outputs=chatbot
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f1aaecf0656cf3a81e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
